<a href="https://colab.research.google.com/github/gauthamramesh3110/clinical_data_ml/blob/main/CarePlanSuggestorLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch

# Import All Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
home_path = './drive/MyDrive/patient_data_csv/sythea_1k/'

allergies_df = pd.read_csv(home_path + 'allergies.csv')
print(f"Number of allergies: {len(allergies_df)}")

careplans_df = pd.read_csv(home_path + 'careplans.csv')
print(f"Number of careplans: {len(careplans_df)}")

conditions_df = pd.read_csv(home_path + 'conditions.csv')
print(f"Number of conditions: {len(conditions_df)}")

encounters_df = pd.read_csv(home_path + 'encounters.csv')
print(f"Number of encounters: {len(encounters_df)}")

immunizations_df = pd.read_csv(home_path + 'immunizations.csv')
print(f"Number of immunizations: {len(immunizations_df)}")

medications_df = pd.read_csv(home_path + 'medications.csv')
print(f"Number of medications: {len(medications_df)}")

observations_df = pd.read_csv(home_path + 'observations.csv')
print(f"Number of observations: {len(observations_df)}")

procedures_df = pd.read_csv(home_path + 'procedures.csv')
print(f"Number of procedures: {len(procedures_df)}")

patients_df = pd.read_csv(home_path + 'patients.csv')
print(f"Number of patients: {len(patients_df)}")

In [ ]:
display(allergies_df.head(1))
display(careplans_df.head(1))
display(conditions_df.head(1))
display(encounters_df.head(1))
display(immunizations_df.head(1))
display(medications_df.head(1))
display(observations_df.head(1))
display(procedures_df.head(1))
display(patients_df.head(1))


# Data exploration

In [ ]:
# Print avg number of clinical data per patient and plot it

median_allergies = allergies_df.groupby('PATIENT').size().median()
median_careplans = careplans_df.groupby('PATIENT').size().median()
median_conditions = conditions_df.groupby('PATIENT').size().median()
median_encounters = encounters_df.groupby('PATIENT').size().median()
median_immunizations = immunizations_df.groupby('PATIENT').size().median()
median_medications = medications_df.groupby('PATIENT').size().median()
median_observations = observations_df.groupby('PATIENT').size().median()
median_procedures = procedures_df.groupby('PATIENT').size().median()

In [ ]:
# PLot the median data

import matplotlib.pyplot as plt

data = {
  'Allergies': median_allergies,
  'Careplans': median_careplans,
  'Conditions': median_conditions,
  'Encounters': median_encounters,
  'Immunizations': median_immunizations,
  'Medications': median_medications,
  'Observations': median_observations,
  'Procedures': median_procedures
}

plt.rcParams['figure.figsize'] = [15, 5]
plt.bar(data.keys(), data.values())
plt.xlabel('Clinical Data')
plt.ylabel('Median')
plt.title('Median Clinical Data per Patient')
plt.show()

# increase plot width

In [ ]:
# Retain only the important patient cols

patients_df = patients_df[['Id', 'BIRTHDATE', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER']]
patients_df.head()

In [ ]:
# Convert BIRTHDATE to seconds compared to unix epoch

unix_epoch = pd.Timestamp('1970-01-01').date()

patients_df['BIRTHDATE'] = pd.to_datetime(patients_df['BIRTHDATE']).dt.date.apply(lambda x: (x - unix_epoch).total_seconds())
patients_df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

patients_df['BIRTHDATE'] = scaler.fit_transform(patients_df[['BIRTHDATE']])
patients_df.head()

In [ ]:
# Fill NaN values in MARITAL column with 'U'

patients_df.fillna({
  'MARITAL': 'U',
}, inplace=True)

In [ ]:
# Get number of patients with no birthdate

print(f"Number of patients with no birthdate: {len(patients_df[patients_df['BIRTHDATE'].isna()])}")

In [ ]:
patients_df.groupby('MARITAL').count()

In [ ]:
patients_df.groupby('RACE').count()

In [ ]:
patients_df.groupby('ETHNICITY').count()

In [ ]:
patients_df.groupby('GENDER').count()

In [ ]:
# encode values in MARITAL, RACE, ETHNICITY, GENDER

patients_encoded_df = pd.get_dummies(patients_df, columns=['MARITAL', 'RACE', 'ETHNICITY', 'GENDER'], dtype=float)
patients_encoded_df.head()

## Unique Codes for each clinical domain

In [ ]:
# Get the unique codes for clinical data, and its corresponding description

unique_allergies_codes = allergies_df.groupby('CODE')[['CODE', 'DESCRIPTION']]
print(f"Number of unique allergies codes: {len(unique_allergies_codes)}")

unique_careplan_codes = careplans_df.groupby('CODE')[['CODE', 'DESCRIPTION']]
print(f"Number of unique careplan codes: {len(unique_careplan_codes)}")

unique_conditions_codes = conditions_df.groupby('CODE')[['CODE', 'DESCRIPTION']]
print(f"Number of unique conditions codes: {len(unique_conditions_codes)}")

unique_encounters_codes = encounters_df.groupby('CODE')[['CODE', 'DESCRIPTION']]
print(f"Number of unique encounters codes: {len(unique_encounters_codes)}")

unique_immunizations_codes = immunizations_df.groupby('CODE')[['CODE', 'DESCRIPTION']]
print(f"Number of unique immunizations codes: {len(unique_immunizations_codes)}")

unique_medications_codes = medications_df.groupby('CODE')[['CODE', 'DESCRIPTION']]
print(f"Number of unique medications codes: {len(unique_medications_codes)}")

unique_observations_codes = observations_df.groupby('CODE')[['CODE', 'DESCRIPTION']]
print(f"Number of unique observations codes: {len(unique_observations_codes)}")

unique_procedures_codes = procedures_df.groupby('CODE')[['CODE', 'DESCRIPTION']]
print(f"Number of unique procedures codes: {len(unique_procedures_codes)}")

In [ ]:
# Group clinical data by encounter

allergies_of_encounters = pd.get_dummies(allergies_df[['ENCOUNTER', 'CODE']], columns=['CODE'], prefix='ALLERGIES', dtype=float).groupby('ENCOUNTER').max()
print(f"Number of unique allergies of encounters: {len(allergies_of_encounters)}")

careplans_of_encounters = pd.get_dummies(careplans_df[['ENCOUNTER', 'CODE']], columns=['CODE'], prefix='CAREPLANS', dtype=float).groupby('ENCOUNTER').max()
print(f"Number of unique careplans of encounters: {len(careplans_of_encounters)}")

conditions_of_encounters = pd.get_dummies(conditions_df[['ENCOUNTER', 'CODE']], columns=['CODE'], prefix='CONDITIONS', dtype=float).groupby('ENCOUNTER').max()
print(f"Number of unique conditions of encounters: {len(conditions_of_encounters)}")

immunizations_of_encounters = pd.get_dummies(immunizations_df[['ENCOUNTER', 'CODE']], columns=['CODE'], prefix='IMMUNIZATIONS', dtype=float).groupby('ENCOUNTER').max()
print(f"Number of unique immunizations of encounters: {len(immunizations_of_encounters)}")

medications_of_encounters = pd.get_dummies(medications_df[['ENCOUNTER', 'CODE']], columns=['CODE'], prefix='MEDICATIONS', dtype=float).groupby('ENCOUNTER').max()
print(f"Number of unique medications of encounters: {len(medications_of_encounters)}")

observations_of_encounters = pd.get_dummies(observations_df[['ENCOUNTER', 'CODE']], columns=['CODE'], prefix='OBSERVATIONS', dtype=float).groupby('ENCOUNTER').max()
print(f"Number of unique observations of encounters: {len(observations_of_encounters)}")

procedures_of_encounters = pd.get_dummies(procedures_df[['ENCOUNTER', 'CODE']], columns=['CODE'], prefix='PROCEDURES', dtype=float).groupby('ENCOUNTER').max()
print(f"Number of unique procedures of encounters: {len(procedures_of_encounters)}")

In [ ]:
# Merge clinical data with encounters

encounters_merged_df = pd.get_dummies(encounters_df[['Id', 'START', 'PATIENT', 'CODE']], columns=['CODE'], prefix='ENCOUNTERS', dtype=float)
print(f"encounters_df size: {len(encounters_merged_df)}")


encounters_merged_df = encounters_merged_df.merge(careplans_of_encounters, left_on='Id', right_on='ENCOUNTER', how='left', suffixes=('', '_careplans'))
encounters_merged_df = encounters_merged_df.merge(allergies_of_encounters, left_on='Id', right_on='ENCOUNTER', how='left', suffixes=('', '_allergies'))
encounters_merged_df = encounters_merged_df.merge(conditions_of_encounters, left_on='Id', right_on='ENCOUNTER', how='left', suffixes=('', '_conditions'))
encounters_merged_df = encounters_merged_df.merge(immunizations_of_encounters, left_on='Id', right_on='ENCOUNTER', how='left', suffixes=('', '_immunizations'))
encounters_merged_df = encounters_merged_df.merge(medications_of_encounters, left_on='Id', right_on='ENCOUNTER', how='left', suffixes=('', '_medications'))
encounters_merged_df = encounters_merged_df.merge(procedures_of_encounters, left_on='Id', right_on='ENCOUNTER', how='left', suffixes=('', '_procedures'))
encounters_merged_df = encounters_merged_df.merge(observations_of_encounters, left_on='Id', right_on='ENCOUNTER', how='left', suffixes=('', '_observations'))

# Fill all nan values with 0
encounters_merged_df.fillna(0, inplace=True)

# Sort by PATIENT and DATE
encounters_merged_df.sort_values(by=['PATIENT', 'START'], inplace=True, ascending=[True, True])
encounters_merged_df.reset_index(drop=True, inplace=True)

# Drop ID column
encounters_merged_df.drop(columns=['Id'], inplace=True)

print(f"Size of encounters_merged_df: {len(encounters_merged_df)}")
display(encounters_merged_df.head(5))

In [ ]:
# Group careplans_df by CODE, and get the count and the first description for each group

careplans_code_summary = careplans_df.groupby('CODE').agg(
  COUNT=('CODE', 'count'), # Rename 'CODE' to 'count_of_code' to avoid ambiguity
  DESCRIPTION=('DESCRIPTION', 'first')
).sort_values(by='COUNT', ascending=False)

top_20_careplans = careplans_code_summary.head(20)
top_20_careplans

## Calculate Pairwise Co-occurrence Matrix

In [ ]:
careplan_data_cols = [col for col in encounters_merged_df.columns if col.startswith('CAREPLANS_')]
careplans_data_df = encounters_merged_df[careplan_data_cols]
print(f"Number of careplan columns extracted: {len(careplan_data_cols)}")
careplans_data_df.head()

In [ ]:
co_occurrence_matrix = careplans_data_df.T.dot(careplans_data_df)
print(f"Shape of co-occurrence matrix: {co_occurrence_matrix.shape}")
co_occurrence_matrix.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Prepare labels for the heatmap
# Remove 'CAREPLANS_' prefix and map to descriptions
co_occurrence_matrix_labeled = co_occurrence_matrix.copy()

# Re-define careplan_code_descriptions to ensure it's available in this cell's scope
careplan_code_descriptions = careplans_df[['CODE', 'DESCRIPTION']].drop_duplicates().set_index('CODE')['DESCRIPTION'].to_dict()

def get_description(col_name):
    code_str = col_name.replace('CAREPLANS_', '')
    try:
        code_int = int(code_str)
    except ValueError:
        code_int = code_str # Keep as string if not purely numeric
    return careplan_code_descriptions.get(code_int, code_str)

co_occurrence_matrix_labeled.columns = [get_description(col) for col in co_occurrence_matrix_labeled.columns]
co_occurrence_matrix_labeled.index = [get_description(idx) for idx in co_occurrence_matrix_labeled.index]

# Set diagonal values to NaN to focus on off-diagonal co-occurrence
np.fill_diagonal(co_occurrence_matrix_labeled.values, np.nan)

# Plotting the heatmap
plt.figure(figsize=(16, 14))
sns.heatmap(co_occurrence_matrix_labeled, annot=False, cmap='viridis', fmt='g', linewidths=.5)
plt.title('Careplan Co-occurrence Matrix (Excluding Diagonal)', fontsize=18)
plt.xlabel('Careplan', fontsize=14)
plt.ylabel('Careplan', fontsize=14)
plt.xticks(rotation=90, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Convert Date to seconds compared to the UNIX epoch

unix_epoch = pd.Timestamp('1970-01-01').date()

encounters_merged_df['START'] = pd.to_datetime(encounters_merged_df['START']).dt.date.apply(lambda x: (x - unix_epoch).total_seconds())
encounters_merged_df.head()

In [ ]:
# Scale DATE value

from sklearn.preprocessing import StandardScaler
date_scaler = StandardScaler()

encounters_merged_df['START'] = date_scaler.fit_transform(encounters_merged_df[['START']])
encounters_merged_df.head()

In [ ]:
# Plot number of encounters per patient using encounters_merged_df and grouping by 'PATIENT'

encounters_per_patient = encounters_merged_df.groupby('PATIENT').size()

# Plot lengths of encounters per patients where patients is in the x axis and the encounter count is in y

import matplotlib.pyplot as plt

plt.scatter(encounters_per_patient.index, encounters_per_patient.values)
plt.xlabel('Patient')
plt.ylabel('Number of Encounters')
plt.title('Number of Encounters per Patient')

In [ ]:
patient_encounters = []
patient_encounters_lengths = []
patients = []
patient_careplans = []

careplan_cols = [col for col in encounters_merged_df.columns if col.startswith('CAREPLANS_')]
print(f"There are {len(careplan_cols)} careplan code cols")

In [ ]:
from pandas.core.arrays import period
# Create datasets.
from tqdm import tqdm

# Define top 20 careplan columns based on the previously calculated top_20_careplans
top_20_careplan_codes = top_20_careplans.index.tolist()
top_20_careplan_cols = [f'CAREPLANS_{code}' for code in top_20_careplan_codes]
# Ensure these columns exist in the merged dataframe
top_20_careplan_cols = [col for col in top_20_careplan_cols if col in encounters_merged_df.columns]

print(f"Filtering targets to top {len(top_20_careplan_cols)} careplans.")

patient_without_careplans = 0
patient_without_enough_encounters = 0
for patient_id, patient_encounters_df in tqdm(encounters_merged_df.groupby('PATIENT')):
  # Get all patient_encounters_df records where there is atleast one careplan_col != 0
  patient_careplan_encounters_df = patient_encounters_df[patient_encounters_df[careplan_cols].sum(axis=1) > 0]

  # If there are no care plans for this patient, increment counter and skip
  if len(patient_careplan_encounters_df) == 0:
    patient_without_careplans += 1
    continue

  latest_encouter_with_careplan_series = patient_careplan_encounters_df.tail(1)

  # Extract the scalar START value for comparison to avoid ValueError
  latest_careplan_start_time = latest_encouter_with_careplan_series['START'].iloc[0]

  # Explicitly create a copy to avoid SettingWithCopyWarning
  filtered_encounters_df = patient_encounters_df[patient_encounters_df['START'] <= latest_careplan_start_time].copy()
  if len(filtered_encounters_df) == 0:
    patient_without_enough_encounters += 1
    continue

  # Shift careplan input down by 1
  filtered_encounters_df[careplan_cols] = filtered_encounters_df[careplan_cols].shift(periods=1, axis=0, fill_value=0)
  # Keep only the bottom 500 of the filtered_encounters_df
  filtered_encounters_df = filtered_encounters_df.tail(500)

  patient_encounters_ts = torch.tensor(filtered_encounters_df.drop(columns=['PATIENT']).values, dtype=torch.float32)
  patient_encounters.append(patient_encounters_ts)

  patient_encounters_lengths.append(len(filtered_encounters_df))

  patient_ts = torch.tensor(patients_encoded_df.drop(columns=['Id'])[patients_encoded_df['Id'] == patient_id].iloc[0].values, dtype=torch.float32)
  patients.append(patient_ts)

  # Use only the top 20 careplan columns for the target
  patient_careplan_ts = torch.tensor(latest_encouter_with_careplan_series[top_20_careplan_cols].values, dtype=torch.float32)
  patient_careplans.append(patient_careplan_ts)

print(f"\nNumber of patients without careplans: {patient_without_careplans}")
print(f"Number of patients without enough encounters: {patient_without_enough_encounters}")

In [ ]:
# Print lengths

print(f"Number of patients: {len(patients)}")
print(f"Number of patient_encounters: {len(patient_encounters)}")
print(f"Number of patient_encounters_lengths: {len(patient_encounters_lengths)}")
print(f"Number of patient_careplans: {len(patient_careplans)}")

In [ ]:
# Plot the patient_encounter_length

import matplotlib.pyplot as plt

plt.hist(patient_encounters_lengths, bins=50)
plt.xlabel('Length')
plt.ylabel('Frequency')
plt.title('Patient Encounter Lengths')
plt.show()

In [ ]:
# Average, Max and Min patient encounter size

print(f"Average patient encounter size: {np.median(patient_encounters_lengths)}")
print(f"Max patient encounter size: {max(patient_encounters_lengths)}")
print(f"Min patient encounter size: {min(patient_encounters_lengths)}")

In [ ]:
# Pad sequence

from torch.nn.utils.rnn import pad_sequence

patient_encounters = pad_sequence(patient_encounters, batch_first=True)
patient_encounters_lengths = torch.tensor(patient_encounters_lengths)
patients = torch.stack(patients)
patient_careplans = torch.stack(patient_careplans)
patient_careplans = patient_careplans.squeeze(dim=1)

In [ ]:
# Print shapes

print(f"patient_encounters shape: {patient_encounters.shape}")
print(f"patient_encounters_lengths shape: {patient_encounters_lengths.shape}")
print(f"patients shape: {patients.shape}")
print(f"patient_careplans shape: {patient_careplans.shape}")

Wraps the tensors into a PyTorch Dataset.

Each item is:

(patient_static, encounter_sequence, sequence_length, careplan_label).

In [ ]:
from torch.utils.data import DataLoader, Dataset

class Encounters(Dataset):
  def __init__(
      self,
      patients,
      patient_encounters,
      patient_encounters_lengths,
      patient_careplans
      ):
    self.patients = patients
    self.patient_encounters = patient_encounters
    self.patient_encounters_lengths = patient_encounters_lengths
    self.patient_careplans = patient_careplans

  def __len__(self):
    return len(self.patient_careplans)

  def __getitem__(self, idx):
    return self.patients[idx], self.patient_encounters[idx], self.patient_encounters_lengths[idx], self.patient_careplans[idx]

Instantiates the dataset with your prepared tensors.

In [ ]:
dataset = Encounters(patients, patient_encounters, patient_encounters_lengths, patient_careplans)

Plans an 80/20 split.

Calculates how many samples go into train vs test.

In [ ]:
split_ratio = 0.8

train_size = int(split_ratio * len(dataset))
test_size = len(dataset) - train_size

print(f"Train Size: {train_size}")
print(f"Test Size: {test_size}")

In [ ]:
# Train/Test Split

from torch.utils.data import random_split

seed = torch.Generator().manual_seed(42) #

train_dataset, test_dataset = random_split(
  dataset,
  [train_size, test_size],
  generator=seed
)

In [ ]:
!pip install lightning --quiet

In [ ]:
import torch.nn as nn
import lightning.pytorch as pl
import torchmetrics
from lightning.pytorch.callbacks import EarlyStopping
import torch.nn.functional as F

Conceptually:

Input:

patient static vector → linear layer.

encounter sequence (padded) → LSTM.

Combines:

Last LSTM hidden state (temporal info) + transformed patient vector → fc1 + fc2.

Output:

logits for each of the top-20 careplan labels (multi-label).

Loss:

BCEWithLogitsLoss with pos_weight to handle class imbalance.

Metrics:

MultilabelAccuracy for multi-label prediction.

Optimizer:

Adam + learning rate scheduler that reduces LR when val_loss plateaus.

This is the core clinical trajectory → careplan recommendation model.

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torchmetrics.classification import MultilabelAccuracy
from torch.optim.lr_scheduler import ReduceLROnPlateau

class CarePlanSuggestor(pl.LightningModule):
  def __init__(
      self,
      linear_input_dim,
      lstm_input_dim,
      hidden_dim,
      num_layers,
      output_dim,
      learning_rate=0.0001,
      dropout=0.0,
      threshold=0.5,
      weight_decay=1e-5,
      pos_weight=None # Added pos_weight parameter
    ):
    super().__init__()
    self.save_hyperparameters() # Saves all init args as hyperparameters
    self.linear = nn.Linear(linear_input_dim, hidden_dim)
    self.lstm = nn.LSTM(lstm_input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
    self.fc1 = nn.Linear(hidden_dim*2, hidden_dim)
    self.dropout = nn.Dropout(dropout)
    self.fc2 = nn.Linear(hidden_dim, output_dim)
    self.learning_rate = learning_rate
    self.output_dim = output_dim
    # Use pos_weight if provided
    self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    self.weight_decay = weight_decay
    self.acc_metric = MultilabelAccuracy(num_labels=output_dim, threshold=threshold)

  def forward(self, patient, sequence, sequence_lengths):
    lengths_sorted, sorted_idx = torch.sort(sequence_lengths, descending=True)
    sequence = sequence[sorted_idx]

    packed_sequence = pack_padded_sequence(sequence, lengths_sorted.cpu(), batch_first=True)

    lstm_out, _ = self.lstm(packed_sequence)
    lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True, total_length=sequence.shape[1])

    _, unsort_idx = torch.sort(sorted_idx)
    lstm_out = lstm_out[unsort_idx]

    lstm_out = lstm_out[:, -1, :]
    linear_out = self.linear(patient)

    input_out = torch.cat((linear_out, lstm_out), dim=1)

    output_fc1 = self.fc1(input_out)
    output_dropout = self.dropout(output_fc1)
    output = self.fc2(output_dropout)
    return output

  def training_step(self, batch, batch_idx):
    patients_x, encounters_x, lengths, y = batch
    y_hat = self.forward(patients_x, encounters_x, lengths)
    loss = self.loss_fn(y_hat, y)
    self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)

    acc = self.acc_metric(y_hat, y.int())
    self.log('train_acc', acc, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def validation_step(self, batch, batch_idx):
    patients_x, encounters_x, lengths, y = batch
    y_hat = self.forward(patients_x, encounters_x, lengths)
    loss = self.loss_fn(y_hat, y)
    self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)

    acc = self.acc_metric(y_hat, y.int())
    self.log('val_acc', acc, on_step=False, on_epoch=True, prog_bar=True)

    return loss

  def configure_optimizers(self):
    optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)
    return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"}}


In [ ]:
%load_ext tensorboard

In [ ]:
!rm -rf ./tb_logs # or your specific log directory

Creates a TensorBoardLogger for Lightning so metrics go into tb_logs/my_experiment

In [ ]:
from lightning.pytorch import Trainer
from lightning.pytorch.loggers import TensorBoardLogger

# Create a TensorBoardLogger instance. By default, logs will be saved in 'lightning_logs/'.
logger = TensorBoardLogger("tb_logs", name="my_experiment")

For each careplan class:

pos_weight = (N - positives) / positives if there are positives.

= 0 if no positives (ignore that class).

Helps the loss pay more attention to rare labels.

In [ ]:
# Calculate pos_weight for BCEWithLogitsLoss
num_samples = patient_careplans.shape[0]
num_positive_samples_per_class = patient_careplans.sum(dim=0)

# Avoid division by zero for classes with no positive samples
# If a class has no positive samples, pos_weight will be 0, effectively ignoring it.
pos_weight = torch.where(
    num_positive_samples_per_class > 0,
    (num_samples - num_positive_samples_per_class) / num_positive_samples_per_class,
    torch.zeros_like(num_positive_samples_per_class, dtype=torch.float32)
)

Visualizes how imbalanced each careplan class is.

In [ ]:
# Plot the classs frequencies

import matplotlib.pyplot as plt

plt.bar(range(patient_careplans.shape[1]), num_positive_samples_per_class)
plt.xlabel('Class')
plt.ylabel('Number of positive samples')
plt.title('Class Frequencies')
plt.show()

In [ ]:
print(f"{patients.shape}")
print(f"{patient_encounters.shape}")
print(f"{patient_encounters_lengths.shape}")
print(f"{patient_careplans.shape}")

Wraps datasets into minibatches.

Train loader is shuffled, val loader is not.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=256)

Stops training if val_loss doesn’t improve for 10 epochs.

Prevents overfitting and saves time.

In [ ]:
# Initialize Trainer
# Define the EarlyStopping callback
early_stop_callback = EarlyStopping(
  monitor='val_loss',  # Metric to monitor
  patience=10,         # Number of epochs with no improvement after which training will be stopped
  mode='min'           # 'min' for metrics that should decrease (like loss), 'max' for metrics that should increase (like accuracy)
)

In [ ]:
%tensorboard --logdir tb_logs

Creates CarePlanSuggestor with:

Input dims derived from patients and patient_encounters.

Multi-layer LSTM (6 layers).

Non-zero dropout + weight decay for regularization.

pos_weight clipped to max 10 to avoid extreme weights.

In [ ]:
# Instantiate Model

model = CarePlanSuggestor(
  patients.shape[1],
  patient_encounters.shape[2],
  learning_rate=0.001, # Reduced from 0.01
  hidden_dim=128,       # Reduced from 64 to lower model complexity
  num_layers=6,
  dropout=0.1,         # Increased from 0.2 for stronger regularization
  output_dim=patient_careplans.shape[1],
  pos_weight=torch.clamp(pos_weight, max=10),
  weight_decay=1e-3    # Increased from 1e-5 to penalize large weights
)

Configures Lightning Trainer:

Up to 3000 epochs but early stopping will stop earlier.

Logs to TensorBoard.

Logs metrics every 2 steps.

In [ ]:
trainer = pl.Trainer(
  max_epochs=3000,
  logger=logger,
  log_every_n_steps=2,
  callbacks=[early_stop_callback] # Add the early stopping callback here
)

In [ ]:

# Train Model
trainer.fit(model, train_loader, val_loader)

In [ ]:
import os
import re
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import matplotlib.pyplot as plt

# Define the base directory where TensorBoard logs are stored
log_dir_base = "tb_logs/my_experiment"

# Find the latest version directory within the experiment logs
versions = [d for d in os.listdir(log_dir_base) if re.match(r"version_\d+", d)]
versions.sort(key=lambda x: int(x.split('_')[1]))

if not versions:
    print(f"No TensorBoard log versions found in {log_dir_base}.")
else:
    latest_version_dir = os.path.join(log_dir_base, versions[-1])
    print(f"Loading logs from: {latest_version_dir}")

    # Initialize EventAccumulator to read TensorBoard logs
    event_acc = EventAccumulator(latest_version_dir)
    event_acc.Reload()

    # Extract scalar events for training and validation loss
    # The tags 'train_loss' and 'val_loss' are used because on_epoch=True was set in the model's logging.
    try:
        train_loss_events = event_acc.Scalars('train_loss')
        val_loss_events = event_acc.Scalars('val_loss')

        # Prepare data for plotting
        epochs_train = [e.step for e in train_loss_events]
        values_train = [e.value for e in train_loss_events]

        epochs_val = [e.step for e in val_loss_events]
        values_val = [e.value for e in val_loss_events]

        # Plotting the losses
        plt.figure(figsize=(10, 6))
        plt.plot(epochs_train, values_train, label='Training Loss', marker='o', markersize=4)
        plt.plot(epochs_val, values_val, label='Validation Loss', marker='x', markersize=4)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training and Validation Loss Over Epochs')
        plt.legend()
        plt.grid(True)
        plt.show()
    except KeyError as e:
        print(f"Could not find scalar events: {e}. Make sure the trainer has been run and logged data.")



In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Convert predictions to probabilities
probabilities = torch.sigmoid(predictions).cpu().numpy()
actuals_np = actuals.cpu().numpy()

# Prepare figure for plotting ROC curves
plt.figure(figsize=(12, 10))

# Ensure careplan_code_descriptions is available (from previous cells)
# If it's not globally available, you might need to re-run the cell where it's defined
# careplan_code_descriptions = careplans_df[['CODE', 'DESCRIPTION']].drop_duplicates().set_index('CODE')['DESCRIPTION'].to_dict()

# Iterate over each careplan label
for i, col_name in enumerate(top_20_careplan_cols):
    # Extract true labels and probabilities for the current class
    y_true_class = actuals_np[:, i]
    y_prob_class = probabilities[:, i]

    # Calculate ROC curve
    fpr, tpr, thresholds = roc_curve(y_true_class, y_prob_class)
    roc_auc = auc(fpr, tpr)

    # Get careplan code and description for the legend
    careplan_code = col_name.replace('CAREPLANS_', '')
    description = careplan_code_descriptions.get(int(careplan_code), f'Code {careplan_code}')

    # Plot the ROC curve
    plt.plot(fpr, tpr, label=f'{description} (AUC = {roc_auc:.2f})')

# Plot the random classifier line
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for Each Careplan')
plt.legend(loc='lower right', fontsize='small')
plt.grid(True)
plt.tight_layout()
plt.show()

Disables gradient computation and runs the model over the entire val set.

Saves raw logits (predictions) and ground truth labels (actuals) for metric computation.



In [ ]:
model.eval()
predictions = []
actuals = []

with torch.no_grad():
  for patients_x, encounters_x, lengths, y_true in val_loader:
    y_pred = model(patients_x.to(model.device), encounters_x.to(model.device), lengths.to(model.device))
    predictions.append(y_pred.cpu())
    actuals.append(y_true.cpu())

predictions = torch.cat(predictions)
actuals = torch.cat(actuals)

print(f"Predictions shape: {predictions.shape}")
print(f"Actuals shape: {actuals.shape}")

In [ ]:
from torchmetrics.classification import MultilabelAccuracy, MultilabelPrecision, MultilabelRecall, MultilabelF1Score

# Assuming `predictions` are logits and `actuals` are one-hot encoded ground truth
# Apply sigmoid to predictions to get probabilities, then threshold to get binary predictions
threshold = 0.5
binary_predictions = (torch.sigmoid(predictions) > threshold).int()

# Convert actuals to int as torchmetrics expects int or long for targets
actuals_int = actuals.int()

num_labels = actuals.shape[1]

# Initialize metrics
accuracy_metric = MultilabelAccuracy(num_labels=num_labels, threshold=threshold)
precision_metric = MultilabelPrecision(num_labels=num_labels, threshold=threshold, average='macro')
recall_metric = MultilabelRecall(num_labels=num_labels, threshold=threshold, average='macro')
f1_metric = MultilabelF1Score(num_labels=num_labels, threshold=threshold, average='macro')

# Calculate metrics
accuracy = accuracy_metric(predictions, actuals_int)
precision = precision_metric(predictions, actuals_int)
recall = recall_metric(predictions, actuals_int)
f1_score = f1_metric(predictions, actuals_int)

print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Validation Precision (Macro): {precision:.4f}")
print(f"Validation Recall (Macro): {recall:.4f}")
print(f"Validation F1-Score (Macro): {f1_score:.4f}")


Computes a 2x2 confusion matrix per label.

Plots a grid of small heatmaps, one per careplan label

In [ ]:
import matplotlib.pyplot as plt
from torchmetrics.classification import MultilabelConfusionMatrix
import seaborn as sns
import math

# Calculate the multilabel confusion matrix
confmat_metric = MultilabelConfusionMatrix(num_labels=num_labels, threshold=threshold)
confmat = confmat_metric(predictions, actuals_int)

# Print the shape of the confusion matrix tensor
print(f"Confusion Matrix shape: {confmat.shape}")

# Calculate the number of rows and columns for the subplot grid
num_labels = confmat.shape[0]
num_cols = 6 # You can adjust this number for better layout
num_rows = math.ceil(num_labels / num_cols)

fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 3, num_rows * 2.5))
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

# Extract the column names for careplans from the original dataframe
careplan_column_names = [col for col in encounters_merged_df.columns if col.startswith('CAREPLANS_')]

for i in range(num_labels):
    ax = axes[i]
    sns.heatmap(confmat[i].cpu().numpy(), annot=True, fmt='g', cmap='Blues',
                xticklabels=['Predicted Negative', 'Predicted Positive'], yticklabels=['Actual Negative', 'Actual Positive'], ax=ax, cbar=False)
    ax.set_title(f'{careplan_column_names[i].replace("CAREPLANS_", "")}', fontsize=8)
    ax.tick_params(axis='x', labelsize=7, rotation=45)
    ax.tick_params(axis='y', labelsize=7, rotation=0)
    ax.set_xlabel('')
    ax.set_ylabel('')

# Hide any unused subplots
for j in range(num_labels, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.suptitle('Confusion Matrices for Each Care Plan Type', y=1.02, fontsize=16)
plt.show()

In [ ]:
def predict_careplan(patient_id, model, patients_encoded_df, encounters_merged_df, careplan_cols, careplan_code_descriptions, threshold=0.7):
  """
  Predicts careplans for a given patient ID.

  Args:
    patient_id (str): The ID of the patient.
    model (CarePlanSuggestor): The trained careplan suggestion model.
    patients_encoded_df (pd.DataFrame): DataFrame containing encoded patient demographics.
    encounters_merged_df (pd.DataFrame): DataFrame containing merged and processed encounter data.
    careplan_cols (list): List of column names corresponding to careplan features.
    careplan_code_descriptions (dict): A dictionary mapping careplan codes to their descriptions.
    threshold (float): Probability threshold for binary classification of careplans.

  Returns:
    dict: A dictionary where keys are predicted careplan codes and values are a dictionary containing
          'probability' and 'description' (if predicted). Returns an empty dictionary if no careplans are predicted.

  Raises:
    ValueError: If the patient ID is not found or no encounters are available.
  """
  # 1. Get Patient Demographics
  patient_data_row = patients_encoded_df[patients_encoded_df['Id'] == patient_id]
  if patient_data_row.empty:
    raise ValueError(f"Patient with ID {patient_id} not found in patients_encoded_df.")

  patient_features_tensor = torch.tensor(patient_data_row.drop(columns=['Id']).iloc[0].values, dtype=torch.float32)

  # 2. Get Patient Encounters
  patient_encounters_df_raw = encounters_merged_df[encounters_merged_df['PATIENT'] == patient_id].copy()

  if patient_encounters_df_raw.empty:
    raise ValueError(f"No encounters found for patient with ID {patient_id}.")

  # Use all available encounters for the patient to form the history for the latest prediction
  patient_history_df = patient_encounters_df_raw.drop(columns=['PATIENT'])

  # Apply shift as done in preprocessing: use previous encounters to predict current. The first row becomes all zeros.
  patient_history_df_shifted = patient_history_df.shift(periods=1, axis=0, fill_value=0)

  sequence_tensor = torch.tensor(patient_history_df_shifted.values, dtype=torch.float32)
  sequence_length = torch.tensor([len(patient_history_df_shifted)], dtype=torch.long)

  # Add batch dimension for model input (batch_size=1)
  patient_features_tensor = patient_features_tensor.unsqueeze(0)
  sequence_tensor = sequence_tensor.unsqueeze(0)

  # 3. Make Prediction
  model.eval() # Set model to evaluation mode
  with torch.no_grad():
    logits = model(
      patient_features_tensor.to(model.device),
      sequence_tensor.to(model.device),
      sequence_length.to(model.device)
    )
    probabilities = torch.sigmoid(logits)
    binary_predictions = (probabilities > threshold).int()

  # 4. Map predictions back to careplan codes and include descriptions
  predicted_careplans = {}
  for i, pred_val in enumerate(binary_predictions.squeeze().cpu().numpy()):
    if pred_val == 1:
      careplan_code = careplan_cols[i].replace('CAREPLANS_', '')
      description = careplan_code_descriptions.get(int(careplan_code), 'Description Not Found')
      predicted_careplans[careplan_code] = {'probability': probabilities.squeeze()[i].item(), 'description': description}

  return predicted_careplans

### Example Usage

In [ ]:
# Create a dictionary mapping careplan codes to their descriptions
careplan_code_descriptions = careplans_df[['CODE', 'DESCRIPTION']].drop_duplicates().set_index('CODE')['DESCRIPTION'].to_dict()

print(f"Number of unique careplan descriptions loaded: {len(careplan_code_descriptions)}")

In [ ]:
# Select a patient ID for demonstration
# Let's pick a random patient from the encoded dataframe as an example, with a fixed random_state for repeatability
example_patient_id = patients_encoded_df['Id'].sample(1, random_state=24).iloc[0]

print(f"Predicting careplans for patient ID: {example_patient_id}")

try:
  predicted_careplans = predict_careplan(
    example_patient_id,
    model,
    patients_encoded_df,
    encounters_merged_df,
    careplan_cols,
    careplan_code_descriptions
  )
  if predicted_careplans:
    print("\nPredicted Careplans and their probabilities and descriptions:")
    for code, details in predicted_careplans.items():
      print(f"  Code: {code}, Probability: {details['probability']:.4f}, Description: {details['description']}")
  else:
    print("\nNo careplans predicted for this patient based on the threshold.")
except ValueError as e:
  print(f"Error: {e}")

## Procedure Recommendation Extension
The following cells extend the notebook to also learn and predict **procedures**
using the same FHIR-based trajectory features and LSTM architecture.

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

print("Building patient-level sequences and labels for PROCEDURES ...")

# Identify procedure label columns and encounter feature columns
procedure_cols = [c for c in encounters_merged_df.columns if c.startswith('PROCEDURES_')]

if len(procedure_cols) == 0:
    raise ValueError("No PROCEDURES_ columns found in encounters_merged_df. "
                     "Please ensure procedures were merged correctly.")

# Use all non-identifier columns as encounter-level features (same style as careplans pipeline)
exclude_cols = ['PATIENT']  # keep START and others as features
encounter_feature_cols_proc = [c for c in encounters_merged_df.columns if c not in exclude_cols]

patient_proc_sequences = []
patient_proc_lengths = []
patient_proc_static = []
patient_proc_labels = []

patients_without_procedures = 0
patients_without_enough_history = 0

for patient_id, df_p in encounters_merged_df.groupby('PATIENT'):
    df_p_sorted = df_p.sort_values('START')

    # Encounters where at least one procedure was performed
    label_mask = df_p_sorted[procedure_cols].sum(axis=1) > 0
    if not label_mask.any():
        patients_without_procedures += 1
        continue

    # Take the latest encounter with a procedure as the label encounter
    label_encounters = df_p_sorted[label_mask]
    label_row = label_encounters.iloc[-1]
    label_time = label_row['START']

    # History up to and including this encounter
    history = df_p_sorted[df_p_sorted['START'] <= label_time]

    # Need at least one encounter before the label for a meaningful trajectory
    if len(history) < 2:
        patients_without_enough_history += 1
        continue

    # Input sequence = all encounters BEFORE the label encounter
    seq_df = history.iloc[:-1]

    # Build encounter sequence tensor
    seq_tensor = torch.tensor(
        seq_df[encounter_feature_cols_proc].values,
        dtype=torch.float32
    )

    # Static patient tensor (demographics)
    pat_row = patients_encoded_df[patients_encoded_df['Id'] == patient_id]
    if pat_row.empty:
        # If patient not found in encoded patients (should be rare), skip
        continue

    pat_features = torch.tensor(
        pat_row.drop(columns=['Id']).values.squeeze().astype('float32'),
        dtype=torch.float32
    )

    # Label vector = procedures at the label encounter
    label_vec = torch.tensor(
        label_row[procedure_cols].values.astype('float32'),
        dtype=torch.float32
    )

    patient_proc_sequences.append(seq_tensor)
    patient_proc_lengths.append(len(seq_tensor))
    patient_proc_static.append(pat_features)
    patient_proc_labels.append(label_vec)

print(f"Patients without any procedures: {patients_without_procedures}")
print(f"Patients without enough history before procedure: {patients_without_enough_history}")
print(f"Patients used for procedure model: {len(patient_proc_labels)}")

if len(patient_proc_labels) == 0:
    raise ValueError("No patients found with usable procedure labels and history. "
                     "Cannot train procedure recommendation model.")

# Pad sequences and stack tensors
patient_proc_sequences = pad_sequence(patient_proc_sequences, batch_first=True)
patient_proc_lengths = torch.tensor(patient_proc_lengths, dtype=torch.long)
patient_proc_static = torch.stack(patient_proc_static)
patient_proc_labels = torch.stack(patient_proc_labels)

print("Procedure sequence tensor shape:", patient_proc_sequences.shape)
print("Procedure static patient tensor shape:", patient_proc_static.shape)
print("Procedure label tensor shape:", patient_proc_labels.shape)

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split

class ProcedureEncounters(Dataset):
    """Dataset for procedure recommendation using patient trajectories."""
    def __init__(self, patients_static, encounter_sequences, sequence_lengths, labels):
        self.patients_static = patients_static
        self.encounter_sequences = encounter_sequences
        self.sequence_lengths = sequence_lengths
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.patients_static[idx],
            self.encounter_sequences[idx],
            self.sequence_lengths[idx],
            self.labels[idx]
        )

proc_dataset = ProcedureEncounters(
    patient_proc_static,
    patient_proc_sequences,
    patient_proc_lengths,
    patient_proc_labels
)

train_size_proc = int(0.8 * len(proc_dataset))
val_size_proc = len(proc_dataset) - train_size_proc

generator = torch.Generator().manual_seed(42)
train_proc_dataset, val_proc_dataset = random_split(proc_dataset, [train_size_proc, val_size_proc], generator=generator)

train_proc_loader = DataLoader(train_proc_dataset, batch_size=256, shuffle=True)
val_proc_loader = DataLoader(val_proc_dataset, batch_size=256)

print(f"Procedure train size: {len(train_proc_dataset)}, val size: {len(val_proc_dataset)}")

In [ ]:
from torchmetrics.classification import MultilabelAccuracy, MultilabelPrecision, MultilabelRecall, MultilabelF1Score

# Compute class-wise pos_weight for procedures (to handle imbalance)
num_samples_proc = patient_proc_labels.shape[0]
num_positive_per_class_proc = patient_proc_labels.sum(dim=0)

pos_weight_proc = torch.where(
    num_positive_per_class_proc > 0,
    (num_samples_proc - num_positive_per_class_proc) / num_positive_per_class_proc,
    torch.zeros_like(num_positive_per_class_proc)
)

print("Procedure pos_weight shape:", pos_weight_proc.shape)

# Instantiate a second model for procedures using the same architecture
procedure_model = CarePlanSuggestor(
    linear_input_dim=patient_proc_static.shape[1],
    lstm_input_dim=patient_proc_sequences.shape[2],
    hidden_dim=64,
    num_layers=2,
    dropout=0.1,
    output_dim=patient_proc_labels.shape[1],
    learning_rate=0.01,
    weight_decay=1e-4,
    pos_weight=pos_weight_proc
)

procedure_trainer = pl.Trainer(
    max_epochs=3000,
    logger=logger,
    log_every_n_steps=2,
    callbacks=[early_stop_callback]
)

procedure_trainer.fit(procedure_model, train_proc_loader, val_proc_loader)

Fully parallels predict_careplan but for procedures.

Returns predicted procedure codes + probabilities + human descriptions.

In [ ]:
# Map procedure codes to descriptions for interpretability
procedure_code_descriptions = procedures_df[['CODE', 'DESCRIPTION']].drop_duplicates().set_index('CODE')['DESCRIPTION'].to_dict()

def predict_procedures(patient_id, model, patients_encoded_df, encounters_merged_df, threshold=0.3):
    """Predict likely procedures for a given patient using their trajectory."""
    model.eval()

    if patient_id not in encounters_merged_df['PATIENT'].unique():
        raise ValueError(f"Patient {patient_id} not found in encounters_merged_df.")

    # Extract and sort encounters for this patient
    df_p = encounters_merged_df[encounters_merged_df['PATIENT'] == patient_id].sort_values('START')

    # Reuse same encounter feature columns and procedure label columns
    procedure_cols_local = [c for c in df_p.columns if c.startswith('PROCEDURES_')]
    if len(procedure_cols_local) == 0:
        raise ValueError("No PROCEDURES_ columns found for this patient.")

    # We build a trajectory using ALL encounters and let the model infer procedures
    seq_tensor = torch.tensor(
        df_p[encounter_feature_cols_proc].values,
        dtype=torch.float32
    ).unsqueeze(0)  # add batch dimension
    seq_length = torch.tensor([seq_tensor.shape[1]], dtype=torch.long)

    pat_row = patients_encoded_df[patients_encoded_df['Id'] == patient_id]
    if pat_row.empty:
        raise ValueError(f"Patient {patient_id} not found in patients_encoded_df.")
    pat_features = torch.tensor(
        pat_row.drop(columns=['Id']).values.squeeze().astype('float32'),
        dtype=torch.float32
    ).unsqueeze(0)

    device = next(model.parameters()).device
    seq_tensor = seq_tensor.to(device)
    seq_length = seq_length.to(device)
    pat_features = pat_features.to(device)

    with torch.no_grad():
        logits = model(pat_features, seq_tensor, seq_length)
        probs = torch.sigmoid(logits).squeeze(0)

    predicted_indices = (probs > threshold).nonzero(as_tuple=True)[0].tolist()

    predictions = {}
    for idx in predicted_indices:
        col_name = procedure_cols[idx]  # global list from training cell
        code_str = col_name.replace('PROCEDURES_', '')
        try:
            code_int = int(code_str)
        except ValueError:
            code_int = code_str

        desc = procedure_code_descriptions.get(code_int, "Description Not Found")
        predictions[code_str] = {
            "probability": float(probs[idx].item()),
            "description": desc
        }

    return predictions

# Example usage for a random patient (you can change this to a specific Id)
example_patient_id_proc = patients_encoded_df['Id'].sample(1).iloc[0]
print("Example patient for procedure prediction:", example_patient_id_proc)

example_procedure_predictions = predict_procedures(
    example_patient_id_proc,
    procedure_model,
    patients_encoded_df,
    encounters_merged_df,
    threshold=0.5
)

print("Predicted procedures for patient", example_patient_id_proc)
for code, info in example_procedure_predictions.items():
    print(f"Procedure CODE: {code}, prob={info['probability']:.3f}, desc={info['description']}")